# 03 — XGBoost MetaLabeler Training
Backtest → collect trades → train binary classifier

In [ ]:
import sys, os, pickle

sys.path.insert(0, "..")
import pandas as pd, numpy as np, xgboost as xgb
from data.ingestion.rest_client import BitgetRESTClient
from data.ingestion.resampler import OHLCVResampler, Timeframe
from backtest.engine import BacktestEngine
from strategies.mtf_macd import MTF_MACD_Elder
from strategies.meta_labeling import MetaLabeler

cfg = {
    "exchange": {
        "name": "bitget",
        "symbols": ["BTC/USDT"],
        "type": "spot",
        "rate_limit": {"max_requests_per_second": 10},
        "fees": {"maker": 0.0002, "taker": 0.0006, "slippage": 0.0002},
    },
    "risk": {"initial_capital": 10000, "max_position_pct": 0.50},
    "backtest": {
        "walk_forward_folds": 5,
        "min_train_fraction": 0.33,
        "min_signal_exit_bars": 6,
    },
    "strategies": {
        "mtf_macd_elder": {
            "macd": {"fast": 12, "slow": 26, "signal": 9},
            "exit": {
                "trailing_stop_pct": 0.03,
                "atr_stop_mult": 2.0,
                "min_hold_bars": 1,
            },
            "elder_filter": {"require_volume_confirm": False, "allow_shorts": True},
        }
    },
    "features": {"max_window_bars": 500, "min_bars_required": 50},
    "regime": {
        "trending": {"adx_min": 25, "di_ratio_strong": 1.3, "di_ratio_reverse": 0.77},
        "ranging": {"adx_max": 20, "bb_width_max": 0.04, "vol_max": 0.50},
        "volatile": {"atr_mult": 2.0, "vol_absolute": 1.0, "bb_width_min": 0.08},
        "hysteresis_bars": 2,
        "lookback_bars": 100,
    },
    "meta_labeling": {
        "enabled": True,
        "min_confidence": 0.55,
        "training_samples": 1000,
    },
    "data": {"validation": {"max_price_jump_pct": 30}},
}

In [ ]:
client = BitgetRESTClient(cfg)
df = client.fetch_days(timeframe="1h", days=180)
dd = OHLCVResampler.resample_bulk(df, Timeframe.D1)
print(f"{len(df)} 1H bars, {len(dd)} 1D")
engine = BacktestEngine(cfg)
result = engine.run_walk_forward(df, MTF_MACD_Elder, data_1d=dd)
print(
    f"Backtest: {len(result.trades)} trades | Sharpe: {result.metrics.get('sharpe_ratio', 0):.2f} | WR: {result.metrics.get('win_rate', 0):.1f}%"
)

In [ ]:
labeler = MetaLabeler(cfg)
if labeler.train(result.trades):
    d = labeler.get_diagnostics()
    print(
        f"Trained: {d['training_samples']} samples, {d['features_used']} features, val_acc={d['val_accuracy']:.3f}"
    )
    imp = labeler.model.get_booster().get_score(importance_type="gain")
    top = sorted(imp.items(), key=lambda x: x[1], reverse=True)[:15]
    for n, s in top:
        print(f"  {n:30s} {s:>10.0f}")